BotRacerController
===

In this notebook, connect training process viva IPC. You can send start and stop message to that process.

## Import module

import module for this notebook.

In [ ]:
import json
import posix_ipc
import sys
from learning_racer.teleoperate import NotebookBackend
import ipywidgets.widgets as widgets
import yaml
import os

## Input Camera Calibration Matrix

Enter the calibration values (K and D) from the data collection notebook. These will be added to the config.yml for use in undistortion during training.

In [ ]:
fx, fy = 600.79112202, 598.85259306
cx, cy = 626.87737459, 454.59767681
k1 = -0.0043877
k2 = -0.0456547
p1 = 0.05200435  # This replaces p1
p2 = -0.0220991  # This replaces p2

config_path = '/opt/ai-rc-car/config.yml'

def update_config(b):
    # Load existing config
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
    else:
        config = {}
    
    # Add/Update CAMERA_CALIB section
    config['CAMERA_CALIB'] = {
        'K': [[fx.value, 0, cx.value], [0, fy.value, cy.value], [0, 0, 1]],
        'D': [k1.value, k2.value, p1.value, p2.value]
    }
    
    # Save updated config
    with open(config_path, 'w') as f:
        yaml.safe_dump(config, f)
    print("Config updated with calibration values.")

update_button.on_click(update_config)

## Show toggle button

Show the toggle button for controlling learning process. 

In [ ]:
toggle = widgets.ToggleButton(value=False, description='start stop')
validate = widgets.Valid(value=False, description='Status',)
display(toggle)
display(validate)

## Start controll process

This cell is do communication to learning process. 

In [ ]:
status = False


def callback(status):
    validate.value = status
    
backend = NotebookBackend(callback)
backend.start()

flag = False
def do_toggle(change):
    global flag, backend
    flag = not flag
    backend.send_status(flag)
    
    
do_toggle({'new':False})
toggle.observe(do_toggle, names='value')


## Start the Training Process

Click the button below to launch the RL training process in the background. Once started, use the toggle button above to manually start and stop episodes (e.g., press start to begin an episode, stop when the Jetbot crashes or goes off-road). The training listens for these signals via IPC for synchronous control.

In [ ]:
import subprocess
import threading
from ipywidgets import Output

start_training_button = widgets.Button(description='Start Training')
display(start_training_button)

# Create and display the console output widget
console_output = Output()
display(console_output)

def run_training():
    # Clear previous output
    console_output.clear_output()
    
    # Start the subprocess with piped output (combine stderr into stdout)
    p = subprocess.Popen(
        ['racer', 'train', '-robot', 'jetbot', '--config-path', '/opt/ai-rc-car/config.yml', '--vae', '/opt/ai-rc-car/vae_newroad4.torch', '--time-steps','20000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1  # Line-buffered
    )
    
    # Read and display lines in real-time
    while True:
        line = p.stdout.readline()
        if not line:
            break
        with console_output:
            print(line.strip())
    
    # Wait for process to finish and display exit code
    p.wait()
    with console_output:
        print(f"\nTraining process exited with code: {p.returncode}")

def start_training_clicked(b):
    thread = threading.Thread(target=run_training)
    thread.start()

start_training_button.on_click(start_training_clicked)